# Freight Spot Seasonality

This script allows you to plot seasonality charts for Spark30S and Spark25S freight rates.

For a full explanation of how to import our Freight Spot Rates data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/lng-freight/contracts.html

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.

The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
#Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)


# 2. Importing Freight Data


In [ ]:
def list_contracts(access_token):
    """
    Fetch available contracts. Return contract ticker symbols

    # Procedure:

    Do a GET query to /v1.0/contracts/ with a Bearer token authorization HTTP header.
    """
    content = do_api_get_query(uri="/v1.0/contracts/", access_token=access_token)

    print(">>>> All the contracts you can fetch")
    tickers = []
    tick_names = []
    for contract in content["data"]:
        print(contract["fullName"])
        tickers.append(contract["id"])
        tick_names.append(contract["fullName"])

    return tickers,tick_names

def fetch_historical_price_releases(access_token, ticker, limit=4, offset=None, vessel=None):

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)
    
    # '174-2stroke' or '160-tfde'
    if vessel is not None:
        query_params += "&vessel-type={}".format(vessel)
        
        
    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params),
        access_token=access_token,
    )

    my_dict = content['data']
    
    return my_dict


### Formatting into a Pandas DataFrame

Here we define a function that calls the data and formats it into a Pandas Dataframe.

In [ ]:

# Defining the function for calling, storing and formatting the data into a Pandas DataFrame
def freight_df(access_token, ticker, limit, offset=None, vessel=None):

    dict_hist = fetch_historical_price_releases(access_token, ticker, limit, offset, vessel)

    stored_data = {
        "ReleaseDate": [],
        "Ticker": [],
        "PeriodStart": [],
        "USDperday": [],
        "USDperdayMax": [],
        "USDperdayMin": [],
    }

    for release in dict_hist:

        data_points = release["data"][0]["dataPoints"]

        for data_point in data_points:
            stored_data['Ticker'].append(release["contractId"])
            stored_data['ReleaseDate'].append(release["releaseDate"])

            period_start_at = data_point["deliveryPeriod"]["startAt"]
            stored_data['PeriodStart'].append(period_start_at)

            stored_data['USDperday'].append(data_point["derivedPrices"]["usdPerDay"]["spark"])
            stored_data['USDperdayMin'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMin"])
            stored_data['USDperdayMax'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMax"])
            
    historical_df = pd.DataFrame(stored_data)
        
    historical_df["USDperday"] = pd.to_numeric(historical_df["USDperday"])
    historical_df["USDperdayMax"] = pd.to_numeric(historical_df["USDperdayMax"])
    historical_df["USDperdayMin"] = pd.to_numeric(historical_df["USDperdayMin"])

    historical_df["ReleaseDate"] = pd.to_datetime(historical_df["ReleaseDate"])
        
    return historical_df

# 3. Yearly Comparisons
Pulling Spark30s and Spark25s historical data using the import function defined above

In [ ]:
# Fetching the freight data and storing as dataframe
spark30_174 = freight_df(access_token, ticker='spark30s', limit=1000, vessel='174-2stroke')
spark30_160 = freight_df(access_token, ticker='spark30s', limit=1000, vessel='160-tfde')

spark25_174 = freight_df(access_token, ticker='spark25s', limit=1000, vessel='174-2stroke')
spark25_160 = freight_df(access_token, ticker='spark25s', limit=1000, vessel='160-tfde')

## Sorting Data

We then create new columns to categorise the data we have. 
- By creating the 'Years' column, we can filter the data by year.
- By creating the 'Day of Year' column, we can plot the yearly data on the same x-axis range.
- Similarly, creating the 'Quarters' and 'Seasons' columns

In [ ]:
# Defining function to assing Cals, Seasons and Quarters to each month
def sort_years(df):
    
    reldates = df['ReleaseDate'].to_list()

    years = []
    months = []
    days = []
    monthday = []
    dayofyear = []
    for r in reldates:
        rsplit = r.strftime('%Y-%m-%d').split('-')
        years.append(rsplit[0])
        months.append(rsplit[1])
        days.append(rsplit[2])

        dayofyear.append(r.timetuple().tm_yday)

    df['Year'] = years
    df['Month'] = months
    df['Day'] = days
    df['DayofYear'] = dayofyear
    
    seas_check = [['04','05','06','07','08','09'], ['10','11','12','01','02','03']]
    quart_check = [['01','02','03'],['04','05','06'],['07','08','09'],['10','11','12']]

    seasons = []
    quarters = []

    for i in df['Month'].to_list():
        if i in quart_check[0]:
            quarters.append('Q1')
        if i in quart_check[1]:
            quarters.append('Q2')
        if i in quart_check[2]:
            quarters.append('Q3')
        if i in quart_check[3]:
            quarters.append('Q4')

        if i in seas_check[0]:
            seasons.append('Summer')
        if i in seas_check[1]:
            seasons.append('Winter')

    df['Quarters'] = quarters
    df['Seasons'] = seasons

    return df

In [ ]:
spark30_174 = sort_years(spark30_174)
spark30_160 = sort_years(spark30_160)

spark25_174 = sort_years(spark25_174)
spark25_160 = sort_years(spark25_160)

# 4. Plotting

## 4.1. Spark30S

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style()

# setting up figure
fig, ax = plt.subplots(figsize=(14,7))
year_list = list(spark30_174['Year'].unique())

# Iterating through years and plotting data
for y in range(len(year_list)):
    hdf = spark30_174[spark30_174['Year']==year_list[y]]
    if y != 0:       
        ax.plot(hdf['DayofYear'], hdf['USDperday'], alpha=0.4, label=year_list[y])
    else:
        ax.plot(hdf['DayofYear'], hdf['USDperday'], color = '#48C38D', linewidth=3.0, label=year_list[y])

        ax.plot(hdf['DayofYear'], hdf['USDperdayMin'], color = '#48C38D', alpha=0.1)
        ax.plot(hdf['DayofYear'], hdf['USDperdayMax'], color = '#48C38D', alpha=0.1)
        ax.fill_between(hdf['DayofYear'], hdf['USDperdayMin'], hdf['USDperdayMax'], color = '#48C38D', alpha=0.2)


ax.set_title('Yearly Comparison of Spark30S')
plt.xlabel('Price Release Date')
plt.ylabel('USD per day')
plt.legend()

# Setting custom x-axis ticks location and labels.
xlabels = ['January', 'March', 'May', 'July', 'September', 'November', 'Year End']

# xpos gives the first day of every other month in terms of 'DayofYear'
#xpos_allmonths = [1,32,60,91,121,152,182,213,244,274,305,335]
xpos = [1,60,121,182,244,305,365]

#plt.ylim(0, 100000)

plt.xticks(xpos, xlabels)

current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels(['$ {:,.0f}'.format(x) for x in current_values])


sns.despine(left=True, bottom=True)

plt.grid(True)

## 4.2. Spark25S

In [ ]:
#setting up figure
fig2, ax2 = plt.subplots(figsize=(14,7))
year_list = list(spark25_174['Year'].unique())

# iterating through years and plotting data
for y in range(len(year_list)):
    hdf = spark25_174[spark25_174['Year']==year_list[y]]
    if y != 0:       
        ax2.plot(hdf['DayofYear'], hdf['USDperday'], alpha=0.4, label=year_list[y])
    else:
        ax2.plot(hdf['DayofYear'], hdf['USDperday'], color = '#48C38D', linewidth=3.0, label=year_list[y])

        ax2.plot(hdf['DayofYear'], hdf['USDperdayMin'], color = '#48C38D', alpha=0.1)
        ax2.plot(hdf['DayofYear'], hdf['USDperdayMax'], color = '#48C38D', alpha=0.1)
        ax2.fill_between(hdf['DayofYear'], hdf['USDperdayMin'], hdf['USDperdayMax'], color = '#48C38D', alpha=0.2)


ax2.set_title('Yearly Comparison of Spark25S')
plt.xlabel('Price Release Date')
plt.ylabel('USD per day')
plt.legend()

# Setting custom x-axis ticks location and labels.
xlabels = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', 'Year End']
#xlabels = ['January', 'March', 'May', 'July', 'September', 'November', 'Year End']

# xpos gives the first day of every other month in terms of 'DayofYear'
xpos = [1,32,60,91,121,152,182,213,244,274,305,335, 365]
#xpos = [1,60,121,182,244,305,365]

plt.xticks(xpos, xlabels)

current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels(['$ {:,.0f}'.format(x) for x in current_values])


sns.despine(left=True, bottom=True)
plt.grid(True)